# Zero-shot ROCOv2 captioning --- Qwen3-VL (off-the-shelf, no fine-tuning)

The notebook counterpart of `qwen3vl_roco_zeroshot_v0.py`. It runs **Qwen3-VL**
(instruction-tuned VLM) **zero-shot** on ROCOv2 --- the direct parallel to the
BLIP-2 and BioMedVQA zero-shot runs.

**Same conditions as the other two, for comparability:** the `"a photo of"` prompt and
the same five-metric suite with the **ImageCLEF BERTScore**
(`microsoft/deberta-xlarge-mnli`, F1). Decoding is **greedy** (`num_beams=1`,
`no_repeat_ngram_size=3`, `min/max_new_tokens=8/40`) &mdash; beam=5 was ~8 days on this
GPU; for a clean comparison the BLIP-2/BioMedVQA runs should be re-scored greedy too.

**One necessary adaptation.** Qwen3-VL is instruction-tuned, not a raw LM, so
`"a photo of"` cannot be a plain decoder prefix. We apply it as an **assistant-turn
prefix that the model continues** (the functional analog of priming BLIP-2/GPT-2's
decoder with `"a photo of"`), and the prediction is the **continuation only** --- the
prefix is stripped --- exactly as in the other runs.

### ⚠️ Hardware (TITAN V, 12.6 GB)
Default is **`Qwen3-VL-4B-Instruct` in bf16** (~8&ndash;9 GB): it fits natively and
adds **no quantization confound** &mdash; the clean choice for a report. To try 8B
instead, set `MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"` and `LOAD_IN_4BIT = True`
(~6 GB, but quantized). If you OOM: lower `CAPTION_BATCH` or `MAX_PIXELS`.

> The full 9,927-image run is best done headless via `qwen3vl_roco_zeroshot_v0.py`
> in `tmux`; this notebook defaults to a 500-image subset for a quick number.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # reduce fragmentation
import torch
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if device == "cuda" else torch.float32   # bf16: matches the other runs
print("device =", device, "| dtype =", DTYPE)

In [ ]:
# ------------------------------- Config ------------------------------- #
MODEL_ID     = "/home/matei/qwen3-vl-4b-instruct"   # local copy of 4B bf16 (downloaded once)
LOAD_IN_4BIT = False                          # True + 8B to fit on ~6 GB (quantized)
# Native-resolution clamps (merge unit 32 px -> #tokens ~= area / 1024).
MIN_PIXELS   = 256 * 32 * 32
# 1024 = quality-optimal for ROCO (~99% near-native). A SPEED/VRAM knob: lower to 768/512
# if you want a bigger batch or the run is too slow.
MAX_PIXELS   = 1024 * 32 * 32

In [ ]:
# ------------------------ Load Qwen3-VL (processor + model) ------------------------ #
from transformers import AutoProcessor, AutoModelForImageTextToText
# Concrete class is Qwen3VLForConditionalGeneration; the Auto class resolves to it.
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
# Left-pad: images differ in visual-token count -> batched prompts have unequal
# length, so left padding keeps the shared prompt-length slice valid per batch.
processor.tokenizer.padding_side = "left"

_kwargs = {}
if device == "cuda":
    _kwargs["device_map"] = {"": 0}
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    _kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
else:
    _kwargs["dtype"] = DTYPE
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **_kwargs)
model.eval()
if device == "cuda":
    print(f"VRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ----------------- ROCOv2 test split (same loader as caption_roco.py) ----------------- #
import pandas as pd
ROCO_DIR = "/home/matei/rocov2"
IMG_DIR  = os.path.join(ROCO_DIR, "test")
caps = pd.read_csv(os.path.join(ROCO_DIR, "test_captions.csv")).dropna(subset=["Caption"]).reset_index(drop=True)
records = [{"id": r.ID, "path": os.path.join(IMG_DIR, f"{r.ID}.jpg"), "caption": str(r.Caption)}
           for r in caps.itertuples() if os.path.isfile(os.path.join(IMG_DIR, f"{r.ID}.jpg"))]
print(f"ROCOv2 test: {len(records)} image-caption pairs")
print("example:", records[0]["id"], "->", records[0]["caption"][:80])

In [ ]:
# ------------- Captioning: "a photo of" (assistant-prefix continuation) ------------- #
from tqdm.auto import tqdm
CAPTION_PROMPT = "a photo of"
CAPTION_BATCH  = 1                             # batch>1 rode the VRAM edge and OOM'd on larger images; batching gives ~no speedup here
# GREEDY decoding (num_beams=1, do_sample=False): ~5x faster than beam=5 here (no per-beam
# image re-encoding). Same short cap + n-gram guard as the other runs.
GEN_KWARGS = dict(max_new_tokens=40, min_new_tokens=8, do_sample=False, num_beams=1,
                  no_repeat_ngram_size=3,
                  pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id)
# eos left to generation_config -> stops on <|im_end|>; max_new_tokens=40 caps length.

# Build the prompt text ONCE: user turn = image only, then prime the assistant reply
# with "a photo of" so the model CONTINUES it. The single <|image_pad|> placeholder is
# expanded per-image by the processor at call time (reserved-seat mechanism).
_DUMMY = Image.new("RGB", (32, 32))
_msgs  = [{"role": "user", "content": [{"type": "image", "image": _DUMMY}]}]
PROMPT_TEXT = processor.apply_chat_template(_msgs, tokenize=False,
                                            add_generation_prompt=True) + CAPTION_PROMPT

@torch.no_grad()
def caption_records(recs, batch_size=CAPTION_BATCH):
    preds = []
    for i in tqdm(range(0, len(recs), batch_size), desc="captioning", leave=False):
        chunk = recs[i:i + batch_size]
        imgs  = [Image.open(r["path"]).convert("RGB") for r in chunk]   # radiographs -> RGB
        inputs = processor(text=[PROMPT_TEXT] * len(imgs), images=imgs,
                           padding=True, return_tensors="pt").to(model.device)
        gen = model.generate(**inputs, **GEN_KWARGS)
        gen = gen[:, inputs["input_ids"].shape[1]:]        # left pad -> strip shared prefix
        preds.extend(s.strip() for s in processor.batch_decode(gen, skip_special_tokens=True))
    return preds

In [ ]:
# ------------------------- Eyeball: what does it caption? ------------------------- #
sample = records[:8]
sp = caption_records(sample, batch_size=4)
for r, p in zip(sample, sp):
    print(f"REF : {r['caption'][:95]}")
    print(f"PRED: {p[:95]}\n")

In [ ]:
# ---------------- Caption metrics: BLEU-1..4, METEOR, ROUGE-L, CIDEr, BERTScore ---------------- #
# BERTScore = microsoft/deberta-xlarge-mnli (F1) -- the SAME model the ImageCLEF/ROCOv2
# leaderboard uses, so this is directly comparable to the paper baselines and to the
# BioMedVQA run (roberta-large would sit on a different, higher scale).
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from nltk.translate.meteor_score import meteor_score

# deberta needs three fixes on this stack: (1) unblock its .bin load (weights_only=True,
# safe); (2) cap the sentinel tokenizer max_length; (3) small batch. Built once, reused.
import transformers.modeling_utils as _mu
_mu.check_torch_load_is_safe = lambda *a, **k: None
from bert_score import BERTScorer
BERT_MODEL = "microsoft/deberta-xlarge-mnli"
_bert_scorer = BERTScorer(model_type=BERT_MODEL, batch_size=8)
_bert_scorer._tokenizer.model_max_length = 512

def _norm(s):
    return " ".join(str(s).lower().split())

def compute_caption_metrics(preds, refs):
    # an empty caption crashes BERTScore's empty-string path -> replace with "."
    preds = [p if str(p).strip() else "." for p in preds]
    refs  = [r if str(r).strip() else "." for r in refs]
    gts = {i: [_norm(refs[i])]  for i in range(len(refs))}
    res = {i: [_norm(preds[i])] for i in range(len(preds))}
    bleu, _  = Bleu(4).compute_score(gts, res)
    rouge, _ = Rouge().compute_score(gts, res)
    cider, _ = Cider().compute_score(gts, res)
    meteor = sum(meteor_score([_norm(refs[i]).split()], _norm(preds[i]).split())
                 for i in range(len(preds))) / len(preds)
    _, _, F = _bert_scorer.score(preds, refs, batch_size=8, verbose=False)
    m = {"BLEU-1": bleu[0], "BLEU-2": bleu[1], "BLEU-3": bleu[2], "BLEU-4": bleu[3],
         "METEOR": meteor, "ROUGE-L": rouge, "CIDEr": cider,
         "BERTScore-F1": F.mean().item(), "BERTScore-model": BERT_MODEL}
    print("=" * 48)
    for k, v in m.items():
        print(f"  {k:16s}: {v:.4f}" if isinstance(v, float) else f"  {k:16s}: {v}")
    print("=" * 48)
    return m

In [ ]:
# ------------------------ Zero-shot evaluation on ROCOv2 test ------------------------ #
# Subset for a quick number; set SUBSET_N = None for the full 9,927 test images
# (better run headless via qwen3vl_roco_zeroshot_v0.py in tmux).
SUBSET_N = 500

eval_recs = records if SUBSET_N is None else records[:SUBSET_N]
print(f"captioning {len(eval_recs)} images (prompt={CAPTION_PROMPT!r}) ...")
preds = caption_records(eval_recs)
refs  = [r["caption"] for r in eval_recs]

print("\nQwen3-VL ZERO-SHOT CAPTIONING -- ROCOv2 test")
metrics = compute_caption_metrics(preds, refs)

print("\nexamples:")
for r, p in list(zip(eval_recs, preds))[:5]:
    print(f"  REF : {r['caption'][:100]}")
    print(f"  PRED: {p[:100]}\n")

### Next steps

* Eyeball the sample captions for the plan's failure modes (boilerplate,
  disclaimers, run-on hallucination, RGB issues). Note Qwen3-VL is instruction-tuned,
  so it may still wrap output in prose despite the `"a photo of"` prefix &mdash; that
  is itself an informative comparison point vs the raw-LM models.
* For the headline number, run the full split via `qwen3vl_roco_zeroshot_v0.py` under
  tmux; it writes `roco_qwen3vl_zeroshot_a_photo_of.json` (predictions) and
  `roco_qwen3vl_zeroshot_metrics.json` (metrics), matching the other runs' outputs.
* This is the **zero-shot / "a photo of"** baseline. The radiology-constrained prompt
  (**V1**) is the next variant &mdash; a separate file so both stay comparable.